In [2]:
import os
import json
import numpy as np
import pandas as pd
from shapely.geometry import Polygon, mapping
import fiona

def save_polygon_shapefile_from_mask(csv_path, output_dir, reference_crs="EPSG:26916"):
    """
    Generates a shapefile using polygon coordinates from the 'mask_points_gis' column in the CSV.
    """

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    shapefile_path = os.path.join(output_dir, "polygon_shapes.shp")

    # Load CSV
    df = pd.read_csv(csv_path)

    # Define schema for attributes in shapefile
    schema = {
        'geometry': 'Polygon',
        'properties': {
            'latitude': 'float',
            'longitude': 'float',
            'class': 'int',
            'confidence': 'float',
            'tile_name': 'str:50',
            'area_geo': 'float'
        }
    }

    # Open shapefile writer
    with fiona.open(
        shapefile_path,
        mode='w',
        driver='ESRI Shapefile',
        crs=reference_crs,
        schema=schema
    ) as shp:
        for idx, row in df.iterrows():
            try:
                # Parse mask_points_gis column into a list of coordinate pairs
                mask_str = row.get('mask_points_gis', '[]')
                if not mask_str.strip():
                    continue

                coords = json.loads(mask_str.replace('(', '[').replace(')', ']'))

                # Ensure the coordinates form a closed polygon
                if coords[0] != coords[-1]:
                    coords.append(coords[0])

                if len(coords) < 3:
                    continue  # Not enough points for a valid polygon

                # Create polygon
                poly = Polygon(coords)

                if not poly.is_valid or poly.is_empty:
                    continue

                # Safely get and encode tile name
                tile = str(row.get('tile_name', '')).encode('ascii', 'ignore').decode()[:50]

                # Write the polygon feature
                shp.write({
                    'geometry': mapping(poly),
                    'properties': {
                        'latitude': float(row.get('latitude', 0)),
                        'longitude': float(row.get('longitude', 0)),
                        'class': int(row.get('class', 0)),
                        'confidence': float(row.get('confidence', 0)),
                        'tile_name': tile,
                        'area_geo': float(row.get('area_geo', 0))
                    }
                })

            except Exception as e:
                print(f"⚠️ Skipped row {idx} due to error: {e}")

    print(f"\n✅ Shapefile successfully written to: {shapefile_path}")


save_polygon_shapefile_from_mask("Alabama/Selected_sampled_rows_for_testing.csv", "Alabama_Test/shapefile_output")


✅ Shapefile successfully written to: Alabama_Test/shapefile_output\polygon_shapes.shp


In [34]:
import json
import pandas as pd
import fiona
from shapely.geometry import shape
from shapely.ops import transform as shp_transform
from pyproj import CRS, Transformer

SQM_TO_SQFT = 10.76391041671

def shapefile_to_csv_with_area(
    shp_path: str,
    output_csv: str,
    area_epsg = 26916,    # CRS used to measure area (meters). Use a projected CRS.
    centroid_epsg = 26916 # CRS for centroid_x/centroid_y. Set 4326 for lon/lat.
) -> pd.DataFrame:
    """
    Read a polygon shapefile and write a CSV with:
      - all original attributes
      - area_sqft (computed in area_epsg)
      - centroid_x, centroid_y (in centroid_epsg)
      - mask_points_gis: JSON list of coordinates of the exterior ring(s) in source CRS
        - Polygon -> [[x,y], ...]
        - MultiPolygon -> [ [[x,y], ...], [[x,y], ...], ... ]
    """
    with fiona.open(shp_path) as src:
        src_crs = CRS.from_user_input(src.crs_wkt or src.crs)

        to_area = None
        if area_epsg is not None:
            to_area = Transformer.from_crs(src_crs, CRS.from_epsg(area_epsg), always_xy=True)

        to_centroid = None
        if centroid_epsg is not None:
            to_centroid = Transformer.from_crs(src_crs, CRS.from_epsg(centroid_epsg), always_xy=True)

        rows = []
        for feat in src:
            geom_json = feat.get("geometry")
            if not geom_json:
                continue
            geom = shape(geom_json)
            if geom.is_empty:
                continue

            # Area geometry in area_epsg
            g_area = geom if to_area is None else shp_transform(to_area.transform, geom)
            area_m2 = g_area.area
            area_sqft = area_m2 * SQM_TO_SQFT

            # Centroid in centroid_epsg
            c = geom.centroid
            if to_centroid is not None:
                cx, cy = to_centroid.transform(c.x, c.y)
            else:
                cx, cy = c.x, c.y

            # Mask points in the source CRS (same as shapefile)
            if geom.geom_type == "Polygon":
                ext = list(geom.exterior.coords)
                mask_points = [[float(x), float(y)] for x, y in ext]
            elif geom.geom_type == "MultiPolygon":
                mask_points = []
                for poly in geom.geoms:
                    ext = list(poly.exterior.coords)
                    mask_points.append([[float(x), float(y)] for x, y in ext])
            else:
                # Not a polygonal geometry, skip
                continue

            props = dict(feat.get("properties") or {})
            props.update({
                "area_sqft": float(area_sqft),
                "centroid_x": float(cx),
                "centroid_y": float(cy),
                "mask_points_gis": json.dumps(mask_points),
            })
            rows.append(props)

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    return df


In [35]:
# Example:
shapefile_to_csv_with_area(
    "Alabama/shapefile_output/final_filtered_with_proximity_test.shp",
    "Alabama_Test/polygon_areas_filtered.csv",
    area_epsg=26916,      # area in m² -> ft² via UTM16N
    centroid_epsg=26916    # centroid lon/lat in WGS84
)

,latitude,longitude,class,confidence,tile_name,area_geo,area_gis,length,width,area_sqft,centroid_x,centroid_y,mask_points_gis
0,3.430047e+06,607218.018494,1,0.897321,m_3008501_ne_16_030_20211103.tif,2.97,17594.455662,376.385023,50.423051,17594.472689,607218.386857,3.430047e+06,"[[607214.5499999999, 3430054.05], [607217.85, ..."
1,3.430022e+06,607214.692554,1,0.892004,m_3008502_nw_16_030_20211103.tif,3.42,18805.394412,377.029981,56.237642,18805.412611,607213.989468,3.430021e+06,"[[607167.45, 3430034.25], [607167.75, 3430034...."
2,3.429232e+06,618145.317627,1,0.878117,m_3008502_ne_16_030_20211109.tif,0.36,22491.491967,406.748851,64.205505,22491.513733,618146.184027,3.429233e+06,"[[618202.9500000001, 3429243.75], [618203.8500..."
3,3.429291e+06,618153.861731,1,0.874535,m_3008502_ne_16_030_20211109.tif,0.72,21737.803689,405.760144,60.073958,21737.824726,618154.392780,3.429291e+06,"[[618210.15, 3429301.05], [618211.05, 3429301...."
4,3.429348e+06,618161.762604,1,0.874287,m_3008502_ne_16_030_20211109.tif,0.81,22533.148260,405.661063,64.960193,22533.170066,618162.760868,3.429348e+06,"[[618218.8500000001, 3429359.25], [618219.15, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11280,3.879654e+06,387390.780121,1,0.915534,m_3508863_sw_16_030_20211108.tif,12.51,25046.572730,408.610702,68.421545,25046.596968,387391.145347,3.879650e+06,"[[387370.35, 3879712.0500000003], [387373.35, ..."
11281,3.879668e+06,387426.892822,1,0.906745,m_3508863_sw_16_030_20211108.tif,12.96,32584.908636,403.463268,83.658193,32584.940170,387427.186467,3.879668e+06,"[[387411.75, 3879728.85], [387413.55, 3879728...."
11282,3.879684e+06,387471.448132,1,0.901268,m_3508863_sw_16_030_20211108.tif,11.52,35874.302657,370.155523,100.139800,35874.337374,387471.353876,3.879683e+06,"[[387461.55, 3879739.95], [387461.85, 3879739...."
11283,3.873467e+06,385968.810040,1,0.850239,m_3508863_sw_16_030_20211108.tif,3.78,26882.355874,381.488206,79.372085,26882.381890,385969.286213,3.873469e+06,"[[385997.25, 3873517.65], [385998.45, 3873517...."


In [33]:
import os
import pandas as pd
from pathlib import Path

def keep_only_class01(input_csv: str, output_csv: str) -> int:
    """
    Keep rows with class in {0,1}. Drop all others.
    Returns number of rows removed.
    """
    src = Path(input_csv)
    if not src.exists():
        raise FileNotFoundError(input_csv)

    df = pd.read_csv(src, dtype=str, low_memory=False)
    if "class" not in df.columns:
        raise KeyError("Missing 'class' column")

    s = df["class"].astype(str).str.strip()
    cls = pd.to_numeric(s, errors="coerce")
    mask = cls.isin([0, 1])

    out_df = df.loc[mask].copy()
    out_df["class"] = cls.loc[mask].astype(int)

    out_path = Path(output_csv)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = out_path if src.resolve() != out_path.resolve() else out_path.with_suffix(out_path.suffix + ".tmp")
    out_df.to_csv(tmp, index=False)
    if tmp != out_path:
        os.replace(tmp, out_path)

    return int((~mask).sum())

# Example
# removed = keep_only_class01("in.csv", "out.csv")
# print("Rows removed:", removed)



# Example:
stats = keep_only_class01("Alabama_Test/polygon_areas_filtered.csv",
                          "Alabama_Test/polygon_areas_filtered_1.csv")
print(stats)


0
